# Grey matter stability across sessions

Panels for `output_data/fig_anat_stability.svg`. Reads `stability_per_region.tsv`, `volume_trajectories.tsv`, `volume_trajectories_subject.tsv`, `region_networks.tsv` and the per-subject `gm_volumes/` tables; computes nothing heavy.

Volumes are per FreeSurfer region (68 Desikan cortical regions, 14 subcortical structures, 2 cerebellar cortices). Each region is coloured by the Yeo-7 network covering most of it; the volume is always that of the whole region.

The visual language — network colours, the glass-brain key, ordering by stability, separate legend strips — mirrors the connectome figure of `cneuromod.all.connectome_stats`, so the two papers read as one system.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import numpy as np
import pandas as pd
import yaml
from airoh.figures import panel_size

# Panels are placed in the montage at 1:1, so save at the page DPI and never
# pass bbox_inches="tight" — tight cropping resizes the canvas after the
# fact, breaking the 1:1 guarantee. Use layout="constrained" instead to
# reclaim margins inside the fixed canvas.
#
# The DPI comes from `figures:` in invoke.yaml, exported by `run-notebooks`:
# saved pixels must equal figsize × dpi, so hardcoding it here would silently
# break placement the moment the montage was composed at another resolution.
FIGURE_DPI = int(os.environ.get("FIGURE_MONTAGE_DPI", 300))

# Every panel is ~2 in wide at its true size, so matplotlib's 10 pt defaults
# overflow it. Scale the type down once, globally (same sizes as the
# connectome figure).
plt.rcParams.update({
    "font.size": 6, "axes.labelsize": 6, "axes.titlesize": 7,
    "xtick.labelsize": 5, "ytick.labelsize": 5, "legend.fontsize": 6,
    "lines.linewidth": 1.0, "lines.markersize": 3, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.0, "ytick.major.size": 2.0,
    "xtick.minor.width": 0.4, "ytick.minor.width": 0.4,
})

output_dir = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data")).resolve()
source_dir = Path(os.environ.get("SOURCE_DATA_DIR", "../source_data")).resolve()
figures_base = Path(os.environ.get("FIGURES_DIR", output_dir / "figures")).resolve()
project_root = output_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Each notebook writes into its own folder, named after the notebook, under
# the shared figures tree. That folder is also what `run-notebooks` treats as
# the "already ran" marker: if it exists, the notebook is skipped.
figure_dir = figures_base / "fig_anat_stability"
figure_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "invoke.yaml") as handle:
    invoke_config = yaml.safe_load(handle)


def panel(name, default=(2.0, 1.9)):
    """Size in inches for the panel placed as `name` in fig_anat_stability.svg."""
    return panel_size(f"fig_anat_stability/{name}", default)


def save(fig, name):
    fig.savefig(figure_dir / name, dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"✅ Saved {figure_dir / name}")


def save_legend(handles, labels, name, default_size=(2.0, 0.2), ncol=None, **legend_kwargs):
    """Render `handles`/`labels` alone into `name`, a strip placed under its panel."""
    fig = plt.figure(figsize=panel(name, default_size), layout="constrained")
    style = dict(fontsize=5, frameon=False, handlelength=1.2, handletextpad=0.4,
                 columnspacing=1.0, borderaxespad=0, borderpad=0) | legend_kwargs
    fig.legend(handles, labels, loc="center", ncol=ncol or len(handles), **style)
    save(fig, name)


def load_tables(folder, suffix):
    tables = sorted((output_dir / folder).glob(f"sub-*_{suffix}.tsv"))
    return pd.concat([pd.read_csv(path, sep="\t") for path in tables], ignore_index=True)


volumes = load_tables("gm_volumes", "gm_volumes")
region_networks = pd.read_csv(output_dir / "region_networks.tsv", sep="\t")
stability = pd.read_csv(output_dir / "stability_per_region.tsv", sep="\t")
trajectories = pd.read_csv(output_dir / "volume_trajectories.tsv", sep="\t")
trajectories_subject = pd.read_csv(output_dir / "volume_trajectories_subject.tsv", sep="\t")
volumes = volumes.merge(region_networks[["region", "network"]], on="region", how="left")

In [2]:
# One colour per network, shared by every panel and by the connectome figure.
#
# Canonical Yeo-7 literature colours, exactly as in cneuromod.all.connectome_stats
# (and its source, cneuromod.all.qa_figures), so both papers name a network
# with the same colour. Cerebellum and subcortex are not Yeo networks and get
# two off-palette hues. Yeo's own Limbic (#DCF8A4) is near-invisible on white,
# so it is darkened the same way. Nine hues is more than colour alone can
# separate for colour-blind readers, so every panel also names networks by
# position: A's labels and the x ticks of D.
NETWORK_COLORS = {
    "Vis": "#781286",          # Yeo visual (purple)
    "SomMot": "#4682B4",       # Yeo somatomotor (steel blue)
    "DorsAttn": "#00760E",     # Yeo dorsal attention (green)
    "SalVentAttn": "#C43AFA",  # Yeo ventral attention / salience (violet)
    "Limbic": "#B5B54E",       # Yeo limbic, darkened for legibility
    "Cont": "#E69422",         # Yeo frontoparietal control (orange)
    "Default": "#CD3E4E",      # Yeo default mode (red)
    "cerebellum": "#2E8B8B",   # off-palette teal (not a Yeo network)
    "subcortex": "#8B5E3C",    # off-palette brown (not a Yeo network)
}

# One network order for the key (A) and the per-network panel (D): most
# to least stable, by the median over regions of the within-subject CV.
# Computed, not hardcoded, so it follows the data. Line panels (B) need no
# order. Caveat for the caption: DorsAttn holds 2 regions and Cont 3, so their
# ranks are fragile.
NETWORK_STABILITY_ORDER = (stability.groupby("network")["intra_subject_cv"].median()
                           .sort_values().index.tolist())
print("Most to least stable:", ", ".join(NETWORK_STABILITY_ORDER))

# Subjects: Okabe-Ito colours and marker shapes, as in the connectome figure.
# Colour means network everywhere else, so subject panels never reuse it.
SUBJECT_COLORS = {
    "sub-01": "#0072B2", "sub-02": "#D55E00", "sub-03": "#009E73",
    "sub-04": "#CC79A7", "sub-05": "#E69F00", "sub-06": "#56B4E9",
}
SUBJECT_MARKERS = {"sub-01": "s", "sub-02": "^", "sub-03": "D",
                   "sub-04": "v", "sub-05": "P", "sub-06": "X"}

# Two pair types, in the connectome figure's colours for "within subject"
# (green) and "between subject" (orange).
WITHIN_COLOR, BETWEEN_COLOR = "#1b9e77", "#d95f02"

# Late sessions only a few participants reach are noisy, and the average
# changes composition there (sub-04 has 5 sessions, the others 13-15). Keep a
# session rank when at least this many participants contribute to it.
MIN_SUBJECTS_PER_SESSION = 5


def add_network_color_bubbles(ax, x_positions, networks, marker_size=8):
    """Draw a network-coloured dot under each x tick of a per-network panel."""
    bubble_transform = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.scatter(x_positions, [-0.035] * len(x_positions), s=marker_size,
               c=[NETWORK_COLORS[network] for network in networks],
               transform=bubble_transform, clip_on=False, zorder=5)
    ax.tick_params(axis="x", pad=6)

Most to least stable: cerebellum, subcortex, DorsAttn, Cont, SomMot, SalVentAttn, Default, Vis, Limbic


In [3]:
# Panel A — network_maps.png: the montage's network key.
#
# Nine sagittal glass brains stacked down the left edge, one per network,
# each filled with that network's colour and named beside it, from most to
# least stable. The masks come from the MNI group atlas (Schaefer 1000/7 +
# Tian S3 + Nettekoven), display only — see `analysis/atlas_maps.py`. They show
# each network's territory, not the Desikan regions assigned to it, which
# exist only in native space. Absent atlas content skips the panel.
from analysis.atlas_maps import atlas_paths, network_mask_images

atlases_root = (Path(invoke_config["datasets"]["cneuromod"]["output_dir"])
                / invoke_config["anat"]["atlases"]["subdataset"])
dseg_path, labels_tsv_path = atlas_paths(project_root / atlases_root)

if dseg_path.is_file() and labels_tsv_path.is_file():
    import matplotlib as mpl
    from nilearn import plotting as nilearn_plotting

    masks = network_mask_images(dseg_path, labels_tsv_path, NETWORK_STABILITY_ORDER)
    # nilearn takes the axes over, so this panel keeps the plain layout engine
    # and trims its own margins.
    fig, axes = plt.subplots(len(masks), 1, figsize=panel("network_maps.png", (0.87, 4.8)))
    fig.subplots_adjust(left=0.19, right=1, bottom=0.005, top=0.995, hspace=0.02)
    for ax, (network, mask_img) in zip(np.atleast_1d(axes), masks.items()):
        nilearn_plotting.plot_glass_brain(
            mask_img, axes=ax, display_mode="x", colorbar=False,
            cmap=mpl.colors.ListedColormap([NETWORK_COLORS[network]]),
            vmin=0.5, vmax=1, plot_abs=False,
        )
        ax.text(-0.04, 0.5, network, transform=ax.transAxes, rotation=90,
                ha="center", va="center", fontsize=5, clip_on=False,
                color=NETWORK_COLORS[network])
    save(fig, "network_maps.png")
else:
    print(f"⚠️  No MNI group atlas at {dseg_path} — run `invoke fetch`. "
          "Skipping network_maps.png.")

✅ Saved /home/pbellec/git/anat.stability_grey_matter/output_data/figures/fig_anat_stability/network_maps.png


In [4]:
# Panels B and C: grey matter volume over sessions, as percent deviation from
# each participant's own mean for that region, averaged over regions. The x
# axis is acquisition order: session dates are not available, so no time
# interval is claimed. Every slope is also printed, for the caption.
from analysis.trajectories import slope_pct_per_session

kept_sessions = sorted(trajectories.loc[
    trajectories["n_subjects"] >= MIN_SUBJECTS_PER_SESSION, "session_rank"].unique())
print(f"Sessions kept (≥{MIN_SUBJECTS_PER_SESSION} participants): "
      f"{kept_sessions[0]}–{kept_sessions[-1]}")

slopes = slope_pct_per_session(trajectories_subject)
print(f"Declining subject x network trajectories: {(slopes < 0).sum()} of {len(slopes)}; "
      f"not declining: {slopes[slopes >= 0].round(3).to_dict()}")


def trajectory_axes(name):
    """One axes with a zero line, sized for `name`, x axis in session order."""
    fig, ax = plt.subplots(figsize=panel(name), layout="constrained")
    ax.axhline(0, color="0.75", linewidth=0.5, zorder=0)
    ax.set_xlabel("session (acquisition order)")
    ax.set_ylabel("GM volume vs own mean (%)")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    return fig, ax

Sessions kept (≥5 participants): 1–13
Declining subject x network trajectories: 52 of 54; not declining: {('sub-04', 'DorsAttn'): 0.038, ('sub-04', 'subcortex'): 0.034}


In [5]:
# Panel B — trajectories_networks.png: one line per network, averaged over
# participants. Colours are keyed anatomically by panel A, so no legend.
fig, ax = trajectory_axes("trajectories_networks.png")
for network in NETWORK_STABILITY_ORDER:
    curve = trajectories[(trajectories["network"] == network)
                         & trajectories["session_rank"].isin(kept_sessions)]
    curve = curve.sort_values("session_rank")
    ax.plot(curve["session_rank"], curve["mean_deviation_pct"], marker="o",
            color=NETWORK_COLORS[network], linewidth=0.9, markersize=1.8)
save(fig, "trajectories_networks.png")

✅ Saved /home/pbellec/git/anat.stability_grey_matter/output_data/figures/fig_anat_stability/trajectories_networks.png


In [6]:
# Panel C — trajectories_subjects.png: one line per participant, averaged over
# all regions (network means weighted by their number of regions), with each
# participant's least-squares line. sub-04 has 5 sessions.
all_regions = (trajectories_subject
               .assign(weighted=lambda table: table["mean_deviation_pct"] * table["n_regions"])
               .groupby(["subject", "session_rank"])[["weighted", "n_regions"]].sum())
all_regions = (all_regions["weighted"] / all_regions["n_regions"]).rename("deviation")

fig, ax = trajectory_axes("trajectories_subjects.png")
for subject, curve in all_regions.groupby(level="subject"):
    sessions = curve.index.get_level_values("session_rank")
    ax.plot(sessions, curve.values, color=SUBJECT_COLORS[subject], linewidth=0.6, alpha=0.6,
            marker=SUBJECT_MARKERS[subject], markersize=2, markerfacecolor="white",
            markeredgewidth=0.5)
    slope, intercept = np.polyfit(sessions, curve.values, 1)
    ax.plot(sessions[[0, -1]], intercept + slope * sessions[[0, -1]],
            color=SUBJECT_COLORS[subject], linewidth=1.2)
save(fig, "trajectories_subjects.png")

subjects_present = sorted(all_regions.index.get_level_values("subject").unique())
subject_handles = [plt.Line2D([], [], color=SUBJECT_COLORS[subject], linewidth=0.6,
                              marker=SUBJECT_MARKERS[subject], markersize=2.5,
                              markerfacecolor="white", markeredgewidth=0.5)
                   for subject in subjects_present]
# Bare subject numbers, with "sub-" said once up front.
subject_labels = [subject.removeprefix("sub-") for subject in subjects_present]
subject_labels[0] = subjects_present[0]
save_legend(subject_handles, subject_labels, "trajectories_subjects_legend.png",
            handlelength=1.0, columnspacing=0.6)

✅ Saved /home/pbellec/git/anat.stability_grey_matter/output_data/figures/fig_anat_stability/trajectories_subjects.png
✅ Saved /home/pbellec/git/anat.stability_grey_matter/output_data/figures/fig_anat_stability/trajectories_subjects_legend.png


In [7]:
# Panel D — cv_bars.png: median within- and between-subject CV per network,
# in stability order, with each region overlaid as a dot. Bars start at zero
# (a bar's length is its value), so no axis break here.
fig, ax = plt.subplots(figsize=panel("cv_bars.png", (6.1, 2.0)), layout="constrained")
positions = np.arange(len(NETWORK_STABILITY_ORDER))
width = 0.36
jitter = np.random.default_rng(0)
for offset, column, color in ((-width / 2, "intra_subject_cv", WITHIN_COLOR),
                              (width / 2, "inter_subject_cv", BETWEEN_COLOR)):
    for position, network in zip(positions, NETWORK_STABILITY_ORDER):
        values = 100 * stability.loc[stability["network"] == network, column]
        ax.bar(position + offset, values.median(), width, color=color, alpha=0.85, zorder=1)
        spread = jitter.uniform(-0.1, 0.1, len(values))
        ax.scatter(position + offset + spread, values, s=2, color="0.15", linewidths=0,
                   zorder=2)
ax.set_xticks(positions, NETWORK_STABILITY_ORDER, rotation=45, ha="right")
ax.set_ylabel("coefficient of variation (%)")
ax.set_xlim(-0.6, len(positions) - 0.4)
add_network_color_bubbles(ax, positions, NETWORK_STABILITY_ORDER)
save(fig, "cv_bars.png")

save_legend([plt.Rectangle((0, 0), 1, 1, color=WITHIN_COLOR),
             plt.Rectangle((0, 0), 1, 1, color=BETWEEN_COLOR),
             plt.Line2D([], [], linestyle="none", marker="o", markersize=1.5, color="0.15")],
            ["within-subject (across sessions)", "between-subject", "region"],
            "cv_bars_legend.png", (6.1, 0.2), handlelength=1.0)

✅ Saved /home/pbellec/git/anat.stability_grey_matter/output_data/figures/fig_anat_stability/cv_bars.png
✅ Saved /home/pbellec/git/anat.stability_grey_matter/output_data/figures/fig_anat_stability/cv_bars_legend.png
